# MoMo-FDVS logical PR12 — tiny restart-safe smoke

This owner-operated notebook uses only committed fictitious/controlled fixtures. It checks transaction preprocessing and fitting, lightweight OCR parsing, one tiny image-model epoch, JSON export/reload, inference, atomic checkpoints, durable synchronization and a complete non-promotable run manifest.

In [ ]:
RUN_PROFILE = "smoke"
TARGET_COMMIT = "cc3c59df047f10905392217d892f452bbd456771"
REPOSITORY_URL = "https://github.com/davidagyekum/momo-fraud-detection.git"
DRIVE_ROOT = "/content/drive/MyDrive/momo-fraud"
VM_ROOT = "/content/momo-work"
NOTEBOOK_PATH = "ml/notebooks/colab/01_tiny_restart_safe_smoke.ipynb"
RESUME_RUN_ID = None
SEED = 20260810
assert RUN_PROFILE == "smoke"
assert len(TARGET_COMMIT) == 40 and TARGET_COMMIT != "REPLACE_WITH_PUSHED_PR12_SHA"

In [ ]:
from pathlib import Path
import subprocess
import sys
from google.colab import drive

drive.mount("/content/drive")
repo = Path(VM_ROOT) / "repo"
repo.parent.mkdir(parents=True, exist_ok=True)
if (repo / ".git").is_dir():
    subprocess.run(["git", "-C", str(repo), "fetch", "--prune", "origin"], check=True)
else:
    subprocess.run(["git", "clone", "--no-checkout", REPOSITORY_URL, str(repo)], check=True)
subprocess.run(["git", "-C", str(repo), "checkout", "--detach", TARGET_COMMIT], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--requirement", str(repo / "ml/requirements-runtime.lock")], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", "--editable", str(repo / "ml")], check=True)

In [ ]:
from momo_fdvs_ml.colab import ColabPaths, colab_preflight_report, repository_state
from momo_fdvs_ml.execution import ExecutionProfile

paths = ColabPaths(drive_root=Path(DRIVE_ROOT), vm_root=Path(VM_ROOT))
preflight = colab_preflight_report(repo, paths=paths, profile=ExecutionProfile.SMOKE, notebook=NOTEBOOK_PATH, require_colab=True)
assert preflight["acquisition_executed"] is False
assert preflight["full_training_executed"] is False

In [ ]:
from momo_fdvs_ml.colab import load_colab_secrets

OPTIONAL_SECRET_NAMES = ()
if OPTIONAL_SECRET_NAMES:
    secret_bundle = load_colab_secrets(OPTIONAL_SECRET_NAMES)
    print({"loaded_secret_names": secret_bundle.names})
else:
    print({"loaded_secret_names": []})

In [ ]:
import json
from momo_fdvs_ml.smoke import run_smoke_flow

outputs = run_smoke_flow(repository_root=repo, vm_root=Path(VM_ROOT), drive_root=Path(DRIVE_ROOT), git_state=repository_state(repo), notebook=NOTEBOOK_PATH, run_id=RESUME_RUN_ID, seed=SEED)
safe_summary = {"run_id": outputs.run_id, "manifest_path": str(outputs.manifest_path), "report_path": str(outputs.report_path), "prediction_digest": outputs.prediction_digest, "resumed": outputs.resumed}
print(json.dumps(safe_summary, indent=2, sort_keys=True))

## Stop boundary

Stop after the smoke manifest validates. Download no dataset, open no locked partition and start no reportable training. Return the non-sensitive preflight/smoke summaries and run-manifest hash to Codex for review.